In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

from tools.redmine import get_issues, get_members, get_versions, get_projects

load_dotenv()

True

In [3]:
llm = ChatOpenAI(
    model="openrouter/auto",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

In [4]:
tools = [get_projects, get_issues, get_members, get_versions]

In [5]:
memory = MemorySaver()

In [6]:
SYSTEM_PROMPT = """You are an intelligent project management assistant connected to Redmine in real time.

You have access to 4 tools:
- get_projects: retrieves the list of all available projects
- get_issues: retrieves tasks with dynamic filters
- get_members: retrieves the members of a project
- get_versions: retrieves the sprints and versions of a project

STRICT RULES:
- ALWAYS call a tool before responding — never invent data.
- If you don’t know the project identifier, first call get_projects.
- For overdue tasks: use due_before with today’s date.
- For urgent tasks: use priority_id = '6'.
- If the tool returns empty data, clearly state it.
"""

In [7]:
agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    system_prompt=SYSTEM_PROMPT
)

In [20]:
from pathlib import Path
from IPython.display import Image, display
import base64

mermaid_graph = agent.get_graph().draw_mermaid()

def mm(graph):
    graphbytes = graph.encode("utf8")
    base64_bytes = base64.urlsafe_b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    display(Image(url='https://mermaid.ink/img/' + base64_string))

mm(mermaid_graph)

In [12]:
config = {"configurable": {"thread_id": "session-1"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What projects are currently active?")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

There are 2 active projects: AI Chatbot Platform and E-Commerce Platform.


In [13]:
config = {"configurable": {"thread_id": "session-1"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What are the overdue tasks for project 'Website Redesign'?")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

I need the project ID for 'Website Redesign'. Please provide it so I can search for overdue tasks. If you don't have it, I can list all available projects for you.


In [16]:
config = {"configurable": {"thread_id": "session-1"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Make a health report about the E-commerce platform project.")
        ]
    },
    config=config
)

print(result["messages"][-1].content)

Here is a health report for the E-Commerce Platform project:

**Tasks:**
*   There are 18 tasks in total.
*   16 tasks are in 'New' status and 2 are 'In Progress'.
*   **Urgent tasks (3):**
    *   Production deployment (assigned to Tarek Mejri, due 2026-06-01)
    *   Stripe payment integration (assigned to Omar Cherif, due 2026-04-28)
    *   Shopping cart API (assigned to Lina Saidi, due 2026-03-30)
*   **High priority tasks (6):**
    *   CI/CD pipeline setup (assigned to Omar Cherif, due 2026-05-26)
    *   React.js storefront UI (assigned to Lina Saidi, due 2026-05-18)
    *   Payment webhook handling (assigned to Omar Cherif, due 2026-05-10)
    *   Order management API (assigned to Lina Saidi, due 2026-05-20)
    *   Product catalog API (assigned to Omar Cherif, due 2026-04-05)
    *   Develop user profile API (assigned to Lina Saidi, due 2026-03-25)
*   **Tasks nearing deadline:**
    *   Cart persistence with Redis (assigned to Omar Cherif, due 2026-04-15)
    *   Product cat